# Fase 4: Machine Learning

El objetivo de esta fase es construir un clasificador binario capaz de predecir la probabilidad de que un usuario salte (`skip = 1`) o escuche completa la canción (`skip = 0`). Con esta información se puede implementar un sistema que permita la toma de deciones de manera automatica. 

Por ejemplo: Si el modelo detecta que vas a saltar las siguientes tres canciones, este mismo las quita de manera discreat para asi poder poner una que no saltarias. Con esto obtendras mejores recomendaciones depenediendo de tu horario y dia de la semana.

Debido a que se esta trabajando con los clusters distribuidos de Spark, se debe de optimizar el procesamiento de los datos mediante un pipeline que prepara todos los datos antes de darselos al modelo.

Los pasos del pipeline son los siguientes:
1. **Features Contextuales del Negocio:** Convertir las variables booleanas para responder de manera numerica a la pregunta. Convirtiendo `true` en un 1 y `false` en 0. 
2. **StringIndexer y OneHotEncoder:** Pasamos a numeros los diferentes valores de las variables. Con StringIndexer se traducen los valores de la columna `"user_id"` y la columna creada `"artist_grouped"`. OneHotEncoder es utilizado de manera consecutiva para añadir nuevas columnas llenadas con 1 o 0 representando si el valor de cada valor (ahora numerico) se encuentra de cada registro.
3. **Reducción de Cardinalidad de Artistas:** Debido a la gigantesta cantidad de artistas agrupamos las entidades para mantener un top 100 y a las demas artistas mantenerlos como `"Other Artist"`.
4. **VectorAssembler:** Se consolidan todos los vectores creados anteriormente en un vector unico de caracteristicas llamada `features`

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [2]:
# Create session
spark = SparkSession.builder \
    .appName("machine_learning") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 16:30:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load DataFrame
df = spark.read.parquet("/app/data/silver/spotify_clean.parquet")

In [4]:
# Make skipped, is_shuffle, and is_offline an int value
df_ml = df.withColumn("label", F.col("skipped").cast(IntegerType())) \
          .withColumn("is_shuffle", F.col("shuffle").cast(IntegerType())) \
          .withColumn("is_offline", F.col("offline").cast(IntegerType()))

In [5]:
# Get the top 100 listened artist 
top_100_artistas_df = df_ml.groupBy("master_metadata_album_artist_name") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(100)

top_100_artists = [row["master_metadata_album_artist_name"] for row in top_100_artistas_df.collect()]

In [6]:
optimized_df_ml = df_ml.withColumn(
    "artist_grouped",
    F.when(F.col("master_metadata_album_artist_name").isin(top_100_artists), 
           F.col("master_metadata_album_artist_name"))
    .otherwise(F.lit("Other Artist"))
)

In [7]:
# Convert the name of the artists to numbers
artist_indexer = StringIndexer(inputCol="artist_grouped", outputCol="artist_indexed", handleInvalid="keep")
artist_encoder = OneHotEncoder(inputCol="artist_indexed", outputCol="artist_encoded", handleInvalid="keep")

In [8]:
# Convert the user to number and then assign their column with 1 or 0 values 
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_indexed")
user_encoder = OneHotEncoder(inputCol="user_indexed", outputCol="user_encoded")

In [9]:
# Do one-hot-encoding to the variables
encoder = OneHotEncoder(inputCols=["hour", "day_of_week"], outputCols=["hour_encoded", "day_encoded"])

In [10]:
# Assemble the vectors for each model
assembler_lr = VectorAssembler(
    inputCols=["user_encoded", "hour_encoded", "day_encoded", "artist_encoded", "is_shuffle", "is_offline"],
    outputCol="features"
)

assembler_gbt = VectorAssembler(
    inputCols=["user_indexed", "hour", "day_of_week", "is_shuffle", "is_offline", "artist_indexed"],
    outputCol="features"
)

In [11]:
# Split the data for train and test
train_data, test_data = optimized_df_ml.randomSplit([0.8, 0.2], seed=42)

# Modelo 1: Regresión Logistica

**Estrategia de Balanceo de Clases:** Ya que el dataset presenta un desbalance natural (mayor cantidad de canciones reproducidas en su totalidad vs skips), calculamos un valor llamado `class_weight`. Esto penaliza con mayor rigor las clasificaciones erróneas de la clase minoritaria dentro de la función de costo del optimizador. Con este valor se penaliza con mayor calificacion a aquellas clasificaciones erroneas que le suceden a las canciones saltadas para evitar overfitting

In [12]:
# Create the object
lr = LogisticRegression(labelCol="label", weightCol="class_weight", maxIter=10)

In [13]:
# Assemble the pipeline
pipeline_lr = Pipeline(stages=[user_indexer, user_encoder, artist_indexer, artist_encoder, encoder, assembler_lr, lr])

In [14]:
# Calculate the proportions of the classes
total_counts = train_data.count()
skips_counts = train_data.filter(F.col("label") == 1).count()

# Calculate the balancing factor
balancing_ratio = (total_counts - skips_counts) / total_counts

# Add the weight column to improve the model
train_data_weighted = train_data.withColumn(
    "class_weight", 
    F.when(F.col("label") == 1, balancing_ratio).otherwise(1.0 - balancing_ratio)
)

test_data_weighted = test_data.withColumn(
    "class_weight", 
    F.when(F.col("label") == 1, balancing_ratio).otherwise(1.0 - balancing_ratio)
)

In [15]:
# Train the model
model_lr = pipeline_lr.fit(train_data_weighted)
predictions_lr = model_lr.transform(train_data_weighted)

# Modelo 2: Gradient-Boost Tree Classifier

Utilizamos Gradient-Boost Tree Clssifier para poder clasificar las respuestas a la pregunta ¿El usuario saltara la canción? Al igual que la regresión logistica responde a la misma pregunta, pero nuestra incognita es si lo lograra con la misma eficacia.

Los parametros utilizados fueron los siguientes
* `maxIter=40`: Limite de 40 árboles base para el ensamble.
* `maxDepth=6`: Profundidad controlada para evitar el overfitting.
* `maxBins=200`: Incremento de contenedores para soportar la cantidad de los 100 artistas.

In [16]:
# Create the model
gbt = GBTClassifier(labelCol="label", maxIter=40, maxDepth=6, maxBins=200, seed=42)

In [17]:
# Assemble the pipeline 
pipeline_gbt = Pipeline(stages=[user_indexer, artist_indexer, assembler_gbt, gbt])

In [18]:
# Set the paramters
gbt = GBTClassifier(
    featuresCol="features", 
    labelCol="label", 
    maxIter=40,
    maxDepth=6,  
    maxBins=200,
    seed=42
)

# Construcción del pipeline con las nuevas transformaciones
pipeline_gbt = Pipeline(stages=[user_indexer, artist_indexer, assembler_gbt, gbt])

# Train the model and predict
model_gbt = pipeline_gbt.fit(train_data_weighted)
predictions_gbt = model_gbt.transform(test_data_weighted)

## Evaluar metricas

In [19]:
# Init spark evaluators
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

In [20]:
# Metrics for logistic regression
auc_lr = evaluator_auc.evaluate(predictions_lr)
acc_lr = evaluator_acc.evaluate(predictions_lr)

In [21]:
# Metrics for gradient-boosted trees
auc_gbt = evaluator_auc.evaluate(predictions_gbt)
acc_gbt = evaluator_acc.evaluate(predictions_gbt)

In [22]:
print("==============================================")
print("   RESULTADOS DE EVALUACIÓN DE MODELOS ML")
print("==============================================")

print(f"Modelo 1: Regresión Logística     -> Accuracy: {acc_lr:.4f} | AUC-ROC: {auc_lr:.4f}")
print(f"Modelo 2: Gradient-Boosted Trees  -> Accuracy: {auc_gbt:.4f} | AUC-ROC: {auc_gbt:.4f}")
print("==============================================")

   RESULTADOS DE EVALUACIÓN DE MODELOS ML
Modelo 1: Regresión Logística     -> Accuracy: 0.6835 | AUC-ROC: 0.7050
Modelo 2: Gradient-Boosted Trees  -> Accuracy: 0.7353 | AUC-ROC: 0.7353


### Interpretación de Resultados

1. **Regresión Logística:**
   El modelo lineal alcanza un `Accuracy` del **68.35%** y un `AUC-ROC` de **0.7050**. Al estar balanceado por pesos de clase con la variable `class_weight` se evita el overfitting ya que se penaliza mas los errores en la prediccion de la clase minoritaria. Este modelo es sumamente ligero y se entreno de manera rapida. Con el porcentaje del `Accuracy` se pude decir que el dia, la hora, si uno esta conectado al internet y las demas variables dentro de nuestro vector de `"features"` aportan elementos para predecir si un usuario va a saltar la canción.

2. **Gradient-Boosted Trees:**
   Este modelo superó consistentemente al baseline establecido por la regresion logistica, elevando el `Accuracy` al **73.53%** y logrando un `AUC-ROC` de **0.7353**. La mejoria sucede debido a las correciones que suceden de manera iterativa sobre las iteraciones pasadas dentro del modelo. Además, existe un mapeo de interacciones que permite reconocer el siguiente ejemplo: Un usuario escucha un artista en especifico a las dos de la mañana con shuffle activado. No solo aprende que en la noche pueden haber mas skips, realmente reconoce que con cierto artista a ciertas horas existe la alta posibilidad que se salte la canción.

4. **Conclusión:**
   Aunque el modelo GBT es el ganador, este mismo requiere un costo computacional y un tiempo de entrenamiento significativamente mayor en comparación con la regresion logistica. En este momento es donde es valido preguntarse si todo ese nuevo coste computacional vale para el rendimiento obtenido. Nuestra opinion es que es totalmente valido ya que existe una gran mejora tanto en el `Accuracy` y el `AUC-ROC`. Ademas, si se manejara mediante batch uno se pude permitir ese mayor costo computacional.